In [1]:
import torch

DEVICE = "cuda" if torch.cuda.is_available() else "cpu"

print("PyTorch :", torch.__version__)
print("GPU disponible :", torch.cuda.is_available())
print("Device :", DEVICE)

if torch.cuda.is_available():
    print(
        "GPU :",
        torch.cuda.get_device_name(0)
    )

PyTorch : 2.10.0+cu128
GPU disponible : True
Device : cuda
GPU : Tesla T4


In [2]:
!pip install -q \
    sentence-transformers \
    faiss-gpu-cu12 \
    pandas \
    numpy

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 48.4/48.4 MB 42.5 MB/s eta 0:00:00:00:0100:01


In [3]:
from pathlib import Path

INPUT_ROOT = Path(
    "/kaggle/input"
)

for path in INPUT_ROOT.rglob("*"):
    if path.is_file():
        print(path)

/kaggle/input/datasets/alyassoulsiham/embedding-benchmark-dataset/chunking_manifest.json
/kaggle/input/datasets/alyassoulsiham/embedding-benchmark-dataset/fixed_1024_chunks.jsonl
/kaggle/input/datasets/alyassoulsiham/embedding-benchmark-dataset/evaluation_30_questions.csv


In [4]:
from pathlib import Path

DATA_DIR = Path(
    "/kaggle/input/datasets/"
    "alyassoulsiham/"
    "embedding-benchmark-dataset"
)

print("Dossier existe :", DATA_DIR.exists())

for path in DATA_DIR.iterdir():
    print(path.name)

Dossier existe : True
chunking_manifest.json
fixed_1024_chunks.jsonl
evaluation_30_questions.csv


In [5]:
import json

CHUNKS_PATH = (
    DATA_DIR
    / "fixed_1024_chunks.jsonl"
)

chunk_records = []

with open(
    CHUNKS_PATH,
    "r",
    encoding="utf-8"
) as file:

    for line in file:
        if not line.strip():
            continue

        record = json.loads(line)
        chunk_records.append(record)

chunk_texts = [
    record["page_content"]
    for record in chunk_records
]

chunk_metadata = [
    record["metadata"]
    for record in chunk_records
]

print("Chunks chargés :", len(chunk_records))
print("Textes chargés :", len(chunk_texts))
print("Métadonnées chargées :", len(chunk_metadata))

print("\nPremier chunk :")
print(chunk_texts[0][:500])

Chunks chargés : 1478
Textes chargés : 1478
Métadonnées chargées : 1478

Premier chunk :
Architecture design patterns that support cost optimization

When you design workload architectures, you should use industry patterns that address common challenges. Patterns can help you make intentional tradeoffs within workloads and optimize for your desired outcome. They can also help mitigate risks that originate from specific problems, which can affect reliability, security, performance, and operations. If not mitigated, risks will eventually increase costs. These patterns are backed by re


In [10]:
assert len(chunk_records) == 1478, (
    f"1478 chunks attendus, "
    f"{len(chunk_records)} trouvés."
)

assert all(
    text.strip()
    for text in chunk_texts
), "Certains chunks sont vides."

print(" Les 1478 chunks Fixed 1024 sont valides")

 Les 1478 chunks Fixed 1024 sont valides


In [11]:
import pandas as pd

EVALUATION_PATH = (
    DATA_DIR
    / "evaluation_30_questions.csv"
)

evaluation_df = pd.read_csv(
    EVALUATION_PATH
)

print(
    "Questions chargées :",
    len(evaluation_df)
)

print("\nRépartition par format :")
print(
    evaluation_df["format"]
    .value_counts()
)

display(
    evaluation_df.head()
)
assert len(evaluation_df) == 30

format_counts = (
    evaluation_df["format"]
    .value_counts()
)

assert format_counts.get("markdown", 0) == 12
assert format_counts.get("html", 0) == 10
assert format_counts.get("pdf", 0) == 8

assert evaluation_df["source_exists"].all()
assert evaluation_df["evidence_exists"].all()

print(" Les 30 questions sont valides")

Questions chargées : 30

Répartition par format :
format
markdown    12
html        10
pdf          8
Name: count, dtype: int64


,id,question,answer,expected_source,gold_evidence,expected_source_canonical,format,source_exists,evidence_exists
0,Q001,Who shares responsibility for the sustainabili...,The responsibility is shared between the cloud...,markdown\well-architected__sustainability__ove...,Sustainability for workloads in the cloud is a...,well-architected__sustainability__overview.md,markdown,True,True
1,Q002,How much less carbon efficient can on-premises...,On-premises workloads can be between 50% and 9...,markdown\well-architected__sustainability__ove...,"running it on-premises, which can be between 5...",well-architected__sustainability__overview.md,markdown,True,True
2,Q003,How do architecture decisions influence the su...,Architecture decisions determine how efficient...,markdown\well-architected__sustainability__ove...,your design decisions directly shape how effic...,well-architected__sustainability__overview.md,markdown,True,True
3,Q004,What is a sustainable cloud workload?,It is a workload designed to minimize unnecess...,markdown\well-architected__sustainability__ove...,A sustainable workload is designed to minimize...,well-architected__sustainability__overview.md,markdown,True,True
4,Q005,Which infrastructure practices can improve bot...,"Right-sizing infrastructure, using autoscaling...",markdown\well-architected__sustainability__ove...,"Right-sizing infrastructure, using autoscaling...",well-architected__sustainability__overview.md,markdown,True,True


 Les 30 questions sont valides


In [12]:
MANIFEST_PATH = (
    DATA_DIR
    / "chunking_manifest.json"
)

with open(
    MANIFEST_PATH,
    "r",
    encoding="utf-8"
) as file:
    manifest = json.load(file)

print(
    json.dumps(
        manifest,
        indent=4,
        ensure_ascii=False
    )
)
assert (
    manifest["selected_chunking_strategy"]
    == "fixed"
)

assert manifest["chunk_size"] == 1024
assert manifest["chunk_overlap"] == 204
assert manifest["number_of_chunks"] == 1478
assert manifest["number_of_questions"] == 30

print(
    " Configuration confirmée : "
    "Fixed 1024, overlap 204"
)

{
    "selected_chunking_strategy": "fixed",
    "chunk_size": 1024,
    "chunk_overlap": 204,
    "overlap_ratio": 0.2,
    "number_of_chunks": 1478,
    "number_of_questions": 30,
    "questions_markdown": 12,
    "questions_html": 10,
    "questions_pdf": 8,
    "selection_reason": "Fixed 1024 selected because Evidence Recall@5 reached 1.0."
}
 Configuration confirmée : Fixed 1024, overlap 204


**Partie 1 — Préparation du benchmark d’embedding**

Cellule 1 — Vérifier les objets déjà chargés

In [14]:
required_objects = [
    "chunk_texts",
    "chunk_metadata",
    "evaluation_df",
    "manifest"
]

missing_objects = [
    object_name
    for object_name in required_objects
    if object_name not in globals()
]

print(
    "Objets manquants :",
    missing_objects
)

assert not missing_objects, (
    "Objets manquants : "
    + ", ".join(missing_objects)
)

assert len(chunk_texts) == 1478
assert len(chunk_metadata) == 1478
assert len(evaluation_df) == 30

print(" Données prêtes pour le benchmark")
print("Chunks :", len(chunk_texts))
print("Questions :", len(evaluation_df))

Objets manquants : []
 Données prêtes pour le benchmark
Chunks : 1478
Questions : 30


Cellule 2 — Vérifier et installer les bibliothèques

In [15]:
import importlib.util
import subprocess
import sys


required_packages = {
    "sentence_transformers": "sentence-transformers",
    "faiss": "faiss-cpu"
}


for module_name, package_name in required_packages.items():

    if importlib.util.find_spec(module_name) is None:

        print(
            f"Installation de {package_name}..."
        )

        subprocess.check_call([
            sys.executable,
            "-m",
            "pip",
            "install",
            "-q",
            package_name
        ])

    else:
        print(
            f"{module_name} est déjà installé."
        )


print(" Bibliothèques disponibles")

sentence_transformers est déjà installé.
faiss est déjà installé.
 Bibliothèques disponibles


Cellule 3 — Imports et vérification du GPU

In [16]:
import gc
import json
import re
import time
import unicodedata
from pathlib import Path

import faiss
import numpy as np
import pandas as pd
import torch

from sentence_transformers import (
    SentenceTransformer
)


DEVICE = (
    "cuda"
    if torch.cuda.is_available()
    else "cpu"
)


print("PyTorch :", torch.__version__)
print("FAISS :", faiss.__version__)
print("Device :", DEVICE)


if torch.cuda.is_available():

    print(
        "GPU :",
        torch.cuda.get_device_name(0)
    )

    print(
        "Mémoire GPU :",
        round(
            torch.cuda.get_device_properties(
                0
            ).total_memory
            / (1024 ** 3),
            2
        ),
        "Go"
    )

PyTorch : 2.10.0+cu128
FAISS : 1.14.1
Device : cuda
GPU : Tesla T4
Mémoire GPU : 14.56 Go


Cellule 4 — Configuration des modèles

In [17]:
EMBEDDING_MODELS = {
    "bge_small": {
        "model_name": (
            "BAAI/bge-small-en-v1.5"
        ),

        "expected_dimension": 384,

        "query_prefix": (
            "Represent this sentence for "
            "searching relevant passages: "
        ),

        "passage_prefix": "",

        "normalize_embeddings": True,

        "batch_size": 32
    },

    "minilm": {
        "model_name": (
            "sentence-transformers/"
            "all-MiniLM-L6-v2"
        ),

        "expected_dimension": 384,

        "query_prefix": "",

        "passage_prefix": "",

        "normalize_embeddings": True,

        "batch_size": 32
    },

    "e5_base": {
        "model_name": (
            "intfloat/e5-base-v2"
        ),

        "expected_dimension": 768,

        "query_prefix": "query: ",

        "passage_prefix": "passage: ",

        "normalize_embeddings": True,

        "batch_size": 32
    }
}


for model_key, model_config in (
    EMBEDDING_MODELS.items()
):

    print("\nModèle :", model_key)

    print(
        "Hugging Face :",
        model_config["model_name"]
    )

    print(
        "Dimension attendue :",
        model_config[
            "expected_dimension"
        ]
    )

    print(
        "Préfixe requête :",
        repr(
            model_config[
                "query_prefix"
            ]
        )
    )

    print(
        "Préfixe passage :",
        repr(
            model_config[
                "passage_prefix"
            ]
        )
    )


Modèle : bge_small
Hugging Face : BAAI/bge-small-en-v1.5
Dimension attendue : 384
Préfixe requête : 'Represent this sentence for searching relevant passages: '
Préfixe passage : ''

Modèle : minilm
Hugging Face : sentence-transformers/all-MiniLM-L6-v2
Dimension attendue : 384
Préfixe requête : ''
Préfixe passage : ''

Modèle : e5_base
Hugging Face : intfloat/e5-base-v2
Dimension attendue : 768
Préfixe requête : 'query: '
Préfixe passage : 'passage: '


**Partie 3 — Dossiers de travail et cache**

Cellule 5 — Créer les dossiers

In [18]:
CACHE_DIR = Path(
    "/kaggle/working/"
    "embedding_cache"
)

RESULTS_DIR = Path(
    "/kaggle/working/"
    "embedding_benchmark_results"
)


CACHE_DIR.mkdir(
    parents=True,
    exist_ok=True
)

RESULTS_DIR.mkdir(
    parents=True,
    exist_ok=True
)


print("Dossier cache :", CACHE_DIR)
print("Dossier résultats :", RESULTS_DIR)

Dossier cache : /kaggle/working/embedding_cache
Dossier résultats : /kaggle/working/embedding_benchmark_results


**Partie 4 — Fonction de préparation des textes**

Cellule 6 — Préparer les passages et les requêtes

In [19]:
def prepare_passages(
    texts: list[str],
    passage_prefix: str
) -> list[str]:
    """
    Ajoute au besoin le préfixe attendu
    par le modèle aux passages.
    """

    return [
        passage_prefix + str(text).strip()
        for text in texts
    ]


def prepare_queries(
    questions: list[str],
    query_prefix: str
) -> list[str]:
    """
    Ajoute au besoin l'instruction ou le préfixe
    attendu par le modèle aux questions.
    """

    return [
        query_prefix + str(question).strip()
        for question in questions
    ]

In [20]:
sample_question = [
    "What is Azure Databricks?"
]

sample_passage = [
    "Azure Databricks is an analytics platform."
]


for model_key, config in (
    EMBEDDING_MODELS.items()
):

    prepared_query = prepare_queries(
        questions=sample_question,
        query_prefix=config[
            "query_prefix"
        ]
    )[0]

    prepared_passage = prepare_passages(
        texts=sample_passage,
        passage_prefix=config[
            "passage_prefix"
        ]
    )[0]

    print("\n", model_key)

    print(
        "Question :",
        prepared_query
    )

    print(
        "Passage :",
        prepared_passage
    )


 bge_small
Question : Represent this sentence for searching relevant passages: What is Azure Databricks?
Passage : Azure Databricks is an analytics platform.

 minilm
Question : What is Azure Databricks?
Passage : Azure Databricks is an analytics platform.

 e5_base
Question : query: What is Azure Databricks?
Passage : passage: Azure Databricks is an analytics platform.


**Partie 5 — Fonction de chargement d’un modèle**

Cellule 7 — Charger proprement un modèle

In [21]:
def load_embedding_model(
    model_name: str,
    device: str
) -> SentenceTransformer:
    """
    Charge un modèle Sentence Transformer
    sur le GPU ou le CPU.
    """

    print(
        "Chargement du modèle :",
        model_name
    )

    model = SentenceTransformer(
        model_name,
        device=device
    )

    print(
        "Dimension détectée :",
        model.get_sentence_embedding_dimension()
    )

    print(
        "Longueur maximale :",
        model.max_seq_length,
        "tokens"
    )

    return model

**Partie 6 — Test rapide de BGE uniquement**

Cellule 8 — Charger BGE

In [22]:
test_model_key = "bge_small"

test_config = EMBEDDING_MODELS[
    test_model_key
]


test_model = load_embedding_model(
    model_name=test_config[
        "model_name"
    ],
    device=DEVICE
)

Chargement du modèle : BAAI/bge-small-en-v1.5


modules.json:   0%|          | 0.00/349 [00:00<?, ?B/s]

config_sentence_transformers.json:   0%|          | 0.00/124 [00:00<?, ?B/s]

README.md: 0.00B [00:00, ?B/s]

sentence_bert_config.json:   0%|          | 0.00/52.0 [00:00<?, ?B/s]

...

tokenizer.json: 0.00B [00:00, ?B/s]

special_tokens_map.json:   0%|          | 0.00/125 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/190 [00:00<?, ?B/s]

Dimension détectée : 384
Longueur maximale : 512 tokens


/tmp/ipykernel_58/2090829687.py:22: FutureWarning: The `get_sentence_embedding_dimension` method has been renamed to `get_embedding_dimension`.
  model.get_sentence_embedding_dimension()


Cellule 9 — Générer deux embeddings de test

In [23]:
test_texts = [
    "Azure provides cloud computing services.",
    "Azure Databricks is an analytics platform."
]


test_vectors = test_model.encode(
    test_texts,
    batch_size=2,
    show_progress_bar=False,
    convert_to_numpy=True,
    normalize_embeddings=True
)


test_vectors = np.asarray(
    test_vectors,
    dtype=np.float32
)


print(
    "Forme :",
    test_vectors.shape
)

print(
    "Dimension :",
    test_vectors.shape[1]
)

print(
    "Normes :",
    np.linalg.norm(
        test_vectors,
        axis=1
    )
)


assert (
    test_vectors.shape[1]
    == test_config[
        "expected_dimension"
    ]
)

assert np.allclose(
    np.linalg.norm(
        test_vectors,
        axis=1
    ),
    1.0,
    atol=1e-4
)


print(
    " BGE fonctionne correctement"
)

Forme : (2, 384)
Dimension : 384
Normes : [1. 1.]
 BGE fonctionne correctement


Cellule 10 — Libérer la mémoire du test

In [24]:
del test_model
del test_vectors

gc.collect()


if torch.cuda.is_available():
    torch.cuda.empty_cache()


print(
    " Mémoire du test libérée"
)

 Mémoire du test libérée


**Partie technique — Benchmark complet**

Cellule 11 — Fonctions de normalisation

In [25]:
import re
import unicodedata
from pathlib import Path


def normalize_for_matching(text: str) -> str:
    """
    Normalise un texte pour comparer les preuves :
    - Unicode ;
    - minuscules ;
    - espaces et retours à la ligne.
    """

    text = unicodedata.normalize(
        "NFKC",
        str(text)
    ).lower()

    text = re.sub(
        r"\s+",
        " ",
        text
    )

    return text.strip()


def canonical_source(source: str) -> str:
    """
    Uniformise les chemins Windows/Linux et conserve
    uniquement le nom du fichier.
    """

    normalized_source = (
        str(source)
        .replace("\\", "/")
    )

    return Path(
        normalized_source
    ).name


def detect_format(source: str) -> str:
    """
    Détermine le format documentaire.
    """

    source = str(source).lower()

    if source.endswith(".md"):
        return "markdown"

    if source.endswith(".html"):
        return "html"

    if source.endswith(".pdf"):
        return "pdf"

    return "unknown"

Cellule 12 — Mesure de chevauchement avec la preuve

In [26]:
def evidence_token_recall(
    retrieved_text: str,
    gold_evidence: str
) -> float:
    """
    Calcule la proportion des tokens de la preuve
    présents dans le chunk récupéré.
    """

    retrieved_tokens = set(
        normalize_for_matching(
            retrieved_text
        ).split()
    )

    evidence_tokens = set(
        normalize_for_matching(
            gold_evidence
        ).split()
    )

    if not evidence_tokens:
        return 0.0

    common_tokens = (
        retrieved_tokens
        & evidence_tokens
    )

    return (
        len(common_tokens)
        / len(evidence_tokens)
    )


def is_source_relevant(
    retrieved_source: str,
    expected_source: str
) -> bool:
    """
    Vérifie si le résultat provient du document attendu.
    """

    return (
        canonical_source(retrieved_source)
        ==
        canonical_source(expected_source)
    )


def is_evidence_relevant(
    retrieved_text: str,
    retrieved_source: str,
    expected_source: str,
    gold_evidence: str,
    threshold: float = 0.60
) -> bool:
    """
    Un résultat est pertinent si :
    1. il vient de la bonne source ;
    2. il contient suffisamment de la preuve attendue.
    """

    if not is_source_relevant(
        retrieved_source=retrieved_source,
        expected_source=expected_source
    ):
        return False

    normalized_text = normalize_for_matching(
        retrieved_text
    )

    normalized_evidence = normalize_for_matching(
        gold_evidence
    )

    if not normalized_evidence:
        return False

    if normalized_evidence in normalized_text:
        return True

    overlap_score = evidence_token_recall(
        retrieved_text=retrieved_text,
        gold_evidence=gold_evidence
    )

    return overlap_score >= threshold

Cellule 13 — Fonctions de métriques

In [27]:
def recall_at_k(
    relevance_flags: list[bool],
    k: int
) -> float:
    """
    Retourne 1 si un résultat pertinent existe
    dans les k premiers résultats, sinon 0.
    """

    return float(
        any(relevance_flags[:k])
    )


def reciprocal_rank(
    relevance_flags: list[bool]
) -> float:
    """
    Retourne l'inverse du rang du premier résultat pertinent.
    """

    for rank, is_relevant in enumerate(
        relevance_flags,
        start=1
    ):
        if is_relevant:
            return 1.0 / rank

    return 0.0

Cellule 14 — Fonction de recherche FAISS

In [28]:
def search_faiss(
    query_vector: np.ndarray,
    index,
    metadata_list: list[dict],
    texts: list[str],
    top_k: int = 10
) -> list[dict]:
    """
    Recherche les chunks les plus proches dans FAISS.
    """

    query_vector = np.asarray(
        query_vector,
        dtype=np.float32
    )

    if query_vector.ndim == 1:
        query_vector = query_vector.reshape(
            1,
            -1
        )

    faiss.normalize_L2(
        query_vector
    )

    effective_k = min(
        top_k,
        index.ntotal
    )

    scores, indices = index.search(
        query_vector,
        effective_k
    )

    results = []

    for rank, (
        chunk_index,
        score
    ) in enumerate(
        zip(
            indices[0],
            scores[0]
        ),
        start=1
    ):

        if chunk_index < 0:
            continue

        metadata = metadata_list[
            int(chunk_index)
        ]

        results.append({
            "rank": rank,
            "score": float(score),
            "chunk_index": int(
                chunk_index
            ),
            "chunk_id": metadata.get(
                "chunk_id"
            ),
            "source": metadata.get(
                "source",
                ""
            ),
            "text": texts[
                int(chunk_index)
            ]
        })

    return results

Cellule 15 — Fonction de cache des embeddings

In [29]:
def get_cache_paths(
    model_key: str
) -> tuple[Path, Path, Path]:

    document_cache_path = (
        CACHE_DIR
        / f"{model_key}_document_embeddings.npy"
    )

    query_cache_path = (
        CACHE_DIR
        / f"{model_key}_query_embeddings.npy"
    )

    metadata_cache_path = (
        CACHE_DIR
        / f"{model_key}_cache_metadata.json"
    )

    return (
        document_cache_path,
        query_cache_path,
        metadata_cache_path
    )

Cellule 16 — Charger ou générer les embeddings

In [30]:
def load_or_generate_embeddings(
    model_key: str,
    model_config: dict,
    texts: list[str],
    questions: list[str],
    device: str
) -> dict:
    """
    Charge les embeddings depuis le cache ou les calcule.

    Retourne :
    - embeddings documents ;
    - embeddings questions ;
    - dimension ;
    - temps de calcul/chargement ;
    - taille des caches ;
    - indication cache utilisé ou non.
    """

    (
        document_cache_path,
        query_cache_path,
        metadata_cache_path
    ) = get_cache_paths(
        model_key
    )

    cache_is_valid = (
        document_cache_path.exists()
        and query_cache_path.exists()
        and metadata_cache_path.exists()
    )

    start_time = time.perf_counter()

    if cache_is_valid:

        print(
            "Cache trouvé pour",
            model_key
        )

        document_vectors = np.load(
            document_cache_path,
            allow_pickle=False
        ).astype(
            np.float32,
            copy=False
        )

        query_vectors = np.load(
            query_cache_path,
            allow_pickle=False
        ).astype(
            np.float32,
            copy=False
        )

        with open(
            metadata_cache_path,
            "r",
            encoding="utf-8"
        ) as file:
            cache_metadata = json.load(
                file
            )

        if document_vectors.shape[0] != len(texts):
            raise ValueError(
                f"Cache documents incompatible pour {model_key}."
            )

        if query_vectors.shape[0] != len(questions):
            raise ValueError(
                f"Cache requêtes incompatible pour {model_key}."
            )

        loaded_from_cache = True
        model = None

    else:

        print(
            "Calcul des embeddings pour",
            model_key
        )

        model = load_embedding_model(
            model_name=model_config[
                "model_name"
            ],
            device=device
        )

        prepared_passages = prepare_passages(
            texts=texts,
            passage_prefix=model_config[
                "passage_prefix"
            ]
        )

        prepared_queries = prepare_queries(
            questions=questions,
            query_prefix=model_config[
                "query_prefix"
            ]
        )

        document_vectors = model.encode(
            prepared_passages,
            batch_size=model_config[
                "batch_size"
            ],
            show_progress_bar=True,
            convert_to_numpy=True,
            normalize_embeddings=model_config[
                "normalize_embeddings"
            ]
        )

        query_vectors = model.encode(
            prepared_queries,
            batch_size=model_config[
                "batch_size"
            ],
            show_progress_bar=True,
            convert_to_numpy=True,
            normalize_embeddings=model_config[
                "normalize_embeddings"
            ]
        )

        document_vectors = np.asarray(
            document_vectors,
            dtype=np.float32
        )

        query_vectors = np.asarray(
            query_vectors,
            dtype=np.float32
        )

        np.save(
            document_cache_path,
            document_vectors
        )

        np.save(
            query_cache_path,
            query_vectors
        )

        cache_metadata = {
            "model_key": model_key,
            "model_name": model_config[
                "model_name"
            ],
            "number_of_documents": len(
                texts
            ),
            "number_of_queries": len(
                questions
            ),
            "embedding_dimension": int(
                document_vectors.shape[1]
            ),
            "normalize_embeddings": model_config[
                "normalize_embeddings"
            ],
            "query_prefix": model_config[
                "query_prefix"
            ],
            "passage_prefix": model_config[
                "passage_prefix"
            ],
            "batch_size": model_config[
                "batch_size"
            ]
        }

        with open(
            metadata_cache_path,
            "w",
            encoding="utf-8"
        ) as file:
            json.dump(
                cache_metadata,
                file,
                indent=4,
                ensure_ascii=False
            )

        loaded_from_cache = False

    elapsed_time = (
        time.perf_counter()
        - start_time
    )

    dimension = int(
        document_vectors.shape[1]
    )

    if dimension != model_config[
        "expected_dimension"
    ]:
        raise ValueError(
            f"Dimension inattendue pour {model_key}: "
            f"{dimension}"
        )

    if document_vectors.shape[0] != len(
        texts
    ):
        raise ValueError(
            "Nombre d'embeddings documents incorrect."
        )

    if query_vectors.shape[0] != len(
        questions
    ):
        raise ValueError(
            "Nombre d'embeddings requêtes incorrect."
        )

    document_norms = np.linalg.norm(
        document_vectors,
        axis=1
    )

    query_norms = np.linalg.norm(
        query_vectors,
        axis=1
    )

    print(
        "Documents :",
        document_vectors.shape
    )

    print(
        "Questions :",
        query_vectors.shape
    )

    print(
        "Norme moyenne documents :",
        round(
            float(document_norms.mean()),
            4
        )
    )

    print(
        "Norme moyenne questions :",
        round(
            float(query_norms.mean()),
            4
        )
    )

    return {
        "document_vectors": document_vectors,
        "query_vectors": query_vectors,
        "dimension": dimension,
        "elapsed_time_seconds": elapsed_time,
        "loaded_from_cache": loaded_from_cache,
        "document_cache_path": document_cache_path,
        "query_cache_path": query_cache_path,
        "metadata_cache_path": metadata_cache_path,
        "document_cache_size_mb": (
            document_cache_path.stat().st_size
            / (1024 ** 2)
        ),
        "query_cache_size_mb": (
            query_cache_path.stat().st_size
            / (1024 ** 2)
        ),
        "mean_document_norm": float(
            document_norms.mean()
        ),
        "mean_query_norm": float(
            query_norms.mean()
        )
    }

**Fonction d’évaluation d’un modèle**

Cellule 17 — Évaluer une configuration d’embedding

In [31]:
def evaluate_embedding_model(
    model_key: str,
    model_config: dict,
    document_vectors: np.ndarray,
    query_vectors: np.ndarray,
    texts: list[str],
    metadata_list: list[dict],
    evaluation_data: pd.DataFrame,
    evidence_threshold: float = 0.60,
    top_k: int = 10
) -> tuple[pd.DataFrame, pd.DataFrame]:
    """
    Construit l'index FAISS et évalue les 30 questions.
    """

    dimension = int(
        document_vectors.shape[1]
    )

    document_vectors = np.asarray(
        document_vectors,
        dtype=np.float32
    ).copy()

    faiss.normalize_L2(
        document_vectors
    )

    index_start = time.perf_counter()

    index = faiss.IndexFlatIP(
        dimension
    )

    index.add(
        document_vectors
    )

    indexing_time = (
        time.perf_counter()
        - index_start
    )

    detail_records = []

    for question_position, (
        _,
        row
    ) in enumerate(
        evaluation_data.iterrows()
    ):

        retrieval_start = (
            time.perf_counter()
        )

        results = search_faiss(
            query_vector=query_vectors[
                question_position
            ],
            index=index,
            metadata_list=metadata_list,
            texts=texts,
            top_k=top_k
        )

        retrieval_time = (
            time.perf_counter()
            - retrieval_start
        )

        source_flags = [
            is_source_relevant(
                retrieved_source=result[
                    "source"
                ],
                expected_source=row[
                    "expected_source"
                ]
            )
            for result in results
        ]

        evidence_flags = [
            is_evidence_relevant(
                retrieved_text=result[
                    "text"
                ],
                retrieved_source=result[
                    "source"
                ],
                expected_source=row[
                    "expected_source"
                ],
                gold_evidence=row[
                    "gold_evidence"
                ],
                threshold=evidence_threshold
            )
            for result in results
        ]

        detail_records.append({
            "model_key": model_key,
            "model_name": model_config[
                "model_name"
            ],
            "question_id": row["id"],
            "question": row["question"],
            "format": row["format"],
            "expected_source": row[
                "expected_source"
            ],

            "evidence_recall_at_1": recall_at_k(
                evidence_flags,
                1
            ),
            "evidence_recall_at_3": recall_at_k(
                evidence_flags,
                3
            ),
            "evidence_recall_at_5": recall_at_k(
                evidence_flags,
                5
            ),
            "evidence_recall_at_10": recall_at_k(
                evidence_flags,
                10
            ),
            "evidence_reciprocal_rank": reciprocal_rank(
                evidence_flags
            ),

            "source_recall_at_1": recall_at_k(
                source_flags,
                1
            ),
            "source_recall_at_3": recall_at_k(
                source_flags,
                3
            ),
            "source_recall_at_5": recall_at_k(
                source_flags,
                5
            ),
            "source_recall_at_10": recall_at_k(
                source_flags,
                10
            ),
            "source_reciprocal_rank": reciprocal_rank(
                source_flags
            ),

            "retrieval_time_seconds": retrieval_time,

            "top_1_source": (
                results[0]["source"]
                if results
                else None
            ),
            "top_1_score": (
                results[0]["score"]
                if results
                else None
            )
        })

    details_df = pd.DataFrame(
        detail_records
    )

    retrieval_times_ms = (
        details_df[
            "retrieval_time_seconds"
        ]
        * 1000
    )

    summary_record = {
        "model_key": model_key,
        "model_name": model_config[
            "model_name"
        ],
        "embedding_dimension": dimension,
        "number_of_chunks": len(texts),
        "number_of_questions": len(
            evaluation_data
        ),

        "evidence_recall_at_1": details_df[
            "evidence_recall_at_1"
        ].mean(),
        "evidence_recall_at_3": details_df[
            "evidence_recall_at_3"
        ].mean(),
        "evidence_recall_at_5": details_df[
            "evidence_recall_at_5"
        ].mean(),
        "evidence_recall_at_10": details_df[
            "evidence_recall_at_10"
        ].mean(),
        "evidence_mrr": details_df[
            "evidence_reciprocal_rank"
        ].mean(),

        "source_recall_at_1": details_df[
            "source_recall_at_1"
        ].mean(),
        "source_recall_at_3": details_df[
            "source_recall_at_3"
        ].mean(),
        "source_recall_at_5": details_df[
            "source_recall_at_5"
        ].mean(),
        "source_recall_at_10": details_df[
            "source_recall_at_10"
        ].mean(),
        "source_mrr": details_df[
            "source_reciprocal_rank"
        ].mean(),

        "mean_retrieval_time_ms": (
            retrieval_times_ms.mean()
        ),
        "p95_retrieval_time_ms": (
            retrieval_times_ms.quantile(
                0.95
            )
        ),
        "faiss_indexing_time_seconds": indexing_time
    }

    summary_df = pd.DataFrame([
        summary_record
    ])

    return (
        summary_df,
        details_df
    )

**Lancement automatique des trois modèles**

Cellule 18 — Boucle complète BGE, MiniLM et E5

In [32]:
benchmark_summary_parts = []
benchmark_detail_parts = []

questions = (
    evaluation_df["question"]
    .astype(str)
    .tolist()
)

benchmark_start_time = (
    time.perf_counter()
)


for model_number, (
    model_key,
    model_config
) in enumerate(
    EMBEDDING_MODELS.items(),
    start=1
):

    print("\n" + "=" * 100)

    print(
        f"Modèle {model_number}/"
        f"{len(EMBEDDING_MODELS)} : "
        f"{model_key}"
    )

    print(
        model_config["model_name"]
    )

    print("=" * 100)

    model_start_time = (
        time.perf_counter()
    )

    embedding_output = (
        load_or_generate_embeddings(
            model_key=model_key,
            model_config=model_config,
            texts=chunk_texts,
            questions=questions,
            device=DEVICE
        )
    )

    summary_df, details_df = (
        evaluate_embedding_model(
            model_key=model_key,
            model_config=model_config,
            document_vectors=embedding_output[
                "document_vectors"
            ],
            query_vectors=embedding_output[
                "query_vectors"
            ],
            texts=chunk_texts,
            metadata_list=chunk_metadata,
            evaluation_data=evaluation_df,
            evidence_threshold=0.60,
            top_k=10
        )
    )

    total_model_time = (
        time.perf_counter()
        - model_start_time
    )

    summary_df[
        "embedding_time_or_cache_load_seconds"
    ] = embedding_output[
        "elapsed_time_seconds"
    ]

    summary_df[
        "loaded_from_cache"
    ] = embedding_output[
        "loaded_from_cache"
    ]

    summary_df[
        "document_cache_size_mb"
    ] = embedding_output[
        "document_cache_size_mb"
    ]

    summary_df[
        "query_cache_size_mb"
    ] = embedding_output[
        "query_cache_size_mb"
    ]

    summary_df[
        "mean_document_norm"
    ] = embedding_output[
        "mean_document_norm"
    ]

    summary_df[
        "mean_query_norm"
    ] = embedding_output[
        "mean_query_norm"
    ]

    summary_df[
        "total_model_time_seconds"
    ] = total_model_time

    summary_df[
        "batch_size"
    ] = model_config[
        "batch_size"
    ]

    summary_df[
        "query_prefix"
    ] = model_config[
        "query_prefix"
    ]

    summary_df[
        "passage_prefix"
    ] = model_config[
        "passage_prefix"
    ]

    benchmark_summary_parts.append(
        summary_df.copy()
    )

    benchmark_detail_parts.append(
        details_df.copy()
    )

    print("\nRésultats :")

    print(
        "Recall@1 :",
        round(
            summary_df[
                "evidence_recall_at_1"
            ].iloc[0],
            4
        )
    )

    print(
        "Recall@5 :",
        round(
            summary_df[
            
                "evidence_recall_at_5"
            ].iloc[0],
            4
        )
    )

    print(
        "MRR :",
        round(
            summary_df[
                "evidence_mrr"
            ].iloc[0],
            4
        )
    )

    print(
        "Temps total :",
        round(
            total_model_time,
            2
        ),
        "secondes"
    )

    # Sauvegarde après chaque modèle
    progress_summary_df = pd.concat(
        benchmark_summary_parts,
        ignore_index=True
    )

    progress_details_df = pd.concat(
        benchmark_detail_parts,
        ignore_index=True
    )

    progress_summary_df.to_csv(
        RESULTS_DIR
        / "embedding_summary_progress.csv",
        index=False,
        encoding="utf-8"
    )

    progress_details_df.to_csv(
        RESULTS_DIR
        / "embedding_details_progress.csv",
        index=False,
        encoding="utf-8"
    )

    print(
        "Progression sauvegardée."
    )

    del embedding_output
    del summary_df
    del details_df

    gc.collect()

    if torch.cuda.is_available():
        torch.cuda.empty_cache()


benchmark_total_time = (
    time.perf_counter()
    - benchmark_start_time
)

print("\n" + "=" * 100)

print(
    " BENCHMARK DES EMBEDDINGS TERMINÉ"
)

print(
    "Temps total :",
    round(
        benchmark_total_time / 60,
        2
    ),
    "minutes"
)

print("=" * 100)


Modèle 1/3 : bge_small
BAAI/bge-small-en-v1.5
Calcul des embeddings pour bge_small
Chargement du modèle : BAAI/bge-small-en-v1.5


Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

BertModel LOAD REPORT from: BAAI/bge-small-en-v1.5
Key                     | Status     |  | 
------------------------+------------+--+-
embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.


Dimension détectée : 384
Longueur maximale : 512 tokens


/tmp/ipykernel_58/2090829687.py:22: FutureWarning: The `get_sentence_embedding_dimension` method has been renamed to `get_embedding_dimension`.
  model.get_sentence_embedding_dimension()


...

Dimension détectée : 768
Longueur maximale : 512 tokens


/tmp/ipykernel_58/2090829687.py:22: FutureWarning: The `get_sentence_embedding_dimension` method has been renamed to `get_embedding_dimension`.
  model.get_sentence_embedding_dimension()


Batches:   0%|          | 0/47 [00:00<?, ?it/s]

Batches:   0%|          | 0/1 [00:00<?, ?it/s]

Documents : (1478, 768)
Questions : (30, 768)
Norme moyenne documents : 1.0
Norme moyenne questions : 1.0

Résultats :
Recall@1 : 0.6667
Recall@5 : 0.9667
MRR : 0.7909
Temps total : 46.39 secondes
Progression sauvegardée.

 BENCHMARK DES EMBEDDINGS TERMINÉ
Temps total : 1.15 minutes


**Fusion et analyse des résultats**

Cellule 19 — Construire les tableaux finaux

In [33]:
embedding_summary_df = pd.concat(
    benchmark_summary_parts,
    ignore_index=True
)

embedding_details_df = pd.concat(
    benchmark_detail_parts,
    ignore_index=True
)


assert len(embedding_summary_df) == 3

assert len(embedding_details_df) == (
    3 * 30
)


print(
    "Modèles évalués :",
    len(embedding_summary_df)
)

print(
    "Lignes détaillées :",
    len(embedding_details_df)
)


comparison_columns = [
    "model_key",
    "model_name",
    "embedding_dimension",

    "evidence_recall_at_1",
    "evidence_recall_at_3",
    "evidence_recall_at_5",
    "evidence_recall_at_10",
    "evidence_mrr",

    "source_recall_at_1",
    "source_recall_at_5",
    "source_mrr",

    "mean_retrieval_time_ms",
    "p95_retrieval_time_ms",

    "embedding_time_or_cache_load_seconds",
    "faiss_indexing_time_seconds",
    "total_model_time_seconds",

    "document_cache_size_mb",
    "loaded_from_cache"
]


embedding_comparison_df = (
    embedding_summary_df[
        comparison_columns
    ]
    .sort_values(
        by=[
            "evidence_recall_at_5",
            "evidence_mrr",
            "embedding_time_or_cache_load_seconds"
        ],
        ascending=[
            False,
            False,
            True
        ]
    )
    .reset_index(
        drop=True
    )
)


display(
    embedding_comparison_df.round(4)
)

Modèles évalués : 3
Lignes détaillées : 90


,model_key,model_name,embedding_dimension,evidence_recall_at_1,evidence_recall_at_3,evidence_recall_at_5,evidence_recall_at_10,evidence_mrr,source_recall_at_1,source_recall_at_5,source_mrr,mean_retrieval_time_ms,p95_retrieval_time_ms,embedding_time_or_cache_load_seconds,faiss_indexing_time_seconds,total_model_time_seconds,document_cache_size_mb,loaded_from_cache
0,e5_base,intfloat/e5-base-v2,768,0.6667,0.8667,0.9667,1.0000,0.7909,0.9000,1.0000,0.9444,0.5482,0.6304,46.3155,0.0036,46.3906,4.3302,False
1,bge_small,BAAI/bge-small-en-v1.5,384,0.6667,0.9000,0.9333,0.9667,0.7900,0.9333,1.0000,0.9667,0.2983,0.3919,13.7794,0.0022,13.8505,2.1652,False
2,minilm,sentence-transformers/all-MiniLM-L6-v2,384,0.5333,0.8000,0.8333,0.9000,0.6734,0.7333,0.9667,0.8261,0.3031,0.3297,7.4446,0.0021,7.5110,2.1652,False


Cellule 20 — Performances par format

In [34]:
embedding_by_format_df = (
    embedding_details_df
    .groupby(
        [
            "model_key",
            "model_name",
            "format"
        ],
        as_index=False
    )
    .agg(
        number_of_questions=(
            "question_id",
            "count"
        ),

        evidence_recall_at_1=(
            "evidence_recall_at_1",
            "mean"
        ),

        evidence_recall_at_3=(
            "evidence_recall_at_3",
            "mean"
        ),

        evidence_recall_at_5=(
            "evidence_recall_at_5",
            "mean"
        ),

        evidence_recall_at_10=(
            "evidence_recall_at_10",
            "mean"
        ),

        evidence_mrr=(
            "evidence_reciprocal_rank",
            "mean"
        ),

        mean_retrieval_time_ms=(
            "retrieval_time_seconds",
            lambda values:
                values.mean() * 1000
        )
    )
)


display(
    embedding_by_format_df.round(4)
)

,model_key,model_name,format,number_of_questions,evidence_recall_at_1,evidence_recall_at_3,evidence_recall_at_5,evidence_recall_at_10,evidence_mrr,mean_retrieval_time_ms
0,bge_small,BAAI/bge-small-en-v1.5,html,10,0.8000,0.9000,1.0000,1.0000,0.8700,0.2804
1,bge_small,BAAI/bge-small-en-v1.5,markdown,12,0.5000,0.9167,0.9167,1.0000,0.7083,0.3232
2,bge_small,BAAI/bge-small-en-v1.5,pdf,8,0.7500,0.8750,0.8750,0.8750,0.8125,0.2833
3,e5_base,intfloat/e5-base-v2,html,10,0.7000,0.9000,1.0000,1.0000,0.8083,0.5531
4,e5_base,intfloat/e5-base-v2,markdown,12,0.5000,0.8333,0.9167,1.0000,0.6994,0.5262
5,e5_base,intfloat/e5-base-v2,pdf,8,0.8750,0.8750,1.0000,1.0000,0.9062,0.5753
6,minilm,sentence-transformers/all-MiniLM-L6-v2,html,10,0.5000,0.9000,1.0000,1.0000,0.6917,0.2934
7,minilm,sentence-transformers/all-MiniLM-L6-v2,markdown,12,0.5833,0.7500,0.7500,0.9167,0.6905,0.3025
8,minilm,sentence-transformers/all-MiniLM-L6-v2,pdf,8,0.5000,0.7500,0.7500,0.7500,0.6250,0.3162


**Sauvegarde complète et téléchargement**

Cellule 21 — Enregistrer tous les fichiers

In [35]:
summary_path = (
    RESULTS_DIR
    / "embedding_benchmark_summary.csv"
)

details_path = (
    RESULTS_DIR
    / "embedding_benchmark_details.csv"
)

comparison_path = (
    RESULTS_DIR
    / "embedding_benchmark_comparison.csv"
)

format_path = (
    RESULTS_DIR
    / "embedding_benchmark_by_format.csv"
)

experiment_manifest_path = (
    RESULTS_DIR
    / "embedding_experiment_manifest.json"
)


embedding_summary_df.to_csv(
    summary_path,
    index=False,
    encoding="utf-8"
)

embedding_details_df.to_csv(
    details_path,
    index=False,
    encoding="utf-8"
)

embedding_comparison_df.to_csv(
    comparison_path,
    index=False,
    encoding="utf-8"
)

embedding_by_format_df.to_csv(
    format_path,
    index=False,
    encoding="utf-8"
)


experiment_manifest = {
    "chunking_strategy": "fixed",
    "chunk_size": 1024,
    "chunk_overlap": 204,
    "number_of_chunks": len(
        chunk_texts
    ),
    "number_of_questions": len(
        evaluation_df
    ),
    "models": EMBEDDING_MODELS,
    "top_k": 10,
    "evidence_threshold": 0.60,
    "device": DEVICE,
    "gpu": (
        torch.cuda.get_device_name(0)
        if torch.cuda.is_available()
        else None
    )
}


with open(
    experiment_manifest_path,
    "w",
    encoding="utf-8"
) as file:
    json.dump(
        experiment_manifest,
        file,
        indent=4,
        ensure_ascii=False
    )


print("Résumé :", summary_path)
print("Détails :", details_path)
print("Comparaison :", comparison_path)
print("Par format :", format_path)
print("Manifeste :", experiment_manifest_path)

Résumé : /kaggle/working/embedding_benchmark_results/embedding_benchmark_summary.csv
Détails : /kaggle/working/embedding_benchmark_results/embedding_benchmark_details.csv
Comparaison : /kaggle/working/embedding_benchmark_results/embedding_benchmark_comparison.csv
Par format : /kaggle/working/embedding_benchmark_results/embedding_benchmark_by_format.csv
Manifeste : /kaggle/working/embedding_benchmark_results/embedding_experiment_manifest.json


Cellule 22 — Créer un ZIP complet

In [36]:
import shutil


FINAL_EXPORT_DIR = Path(
    "/kaggle/working/"
    "embedding_benchmark_complete"
)

if FINAL_EXPORT_DIR.exists():
    shutil.rmtree(
        FINAL_EXPORT_DIR
    )

FINAL_EXPORT_DIR.mkdir(
    parents=True,
    exist_ok=True
)


shutil.copytree(
    CACHE_DIR,
    FINAL_EXPORT_DIR / "embedding_cache"
)

shutil.copytree(
    RESULTS_DIR,
    FINAL_EXPORT_DIR / "results"
)


final_archive_path = shutil.make_archive(
    base_name=(
        "/kaggle/working/"
        "embedding_benchmark_complete"
    ),
    format="zip",
    root_dir=FINAL_EXPORT_DIR
)


print(
    "Archive finale :",
    final_archive_path
)

print(
    "Taille :",
    round(
        Path(final_archive_path)
        .stat()
        .st_size
        / (1024 ** 2),
        2
    ),
    "Mo"
)

Archive finale : /kaggle/working/embedding_benchmark_complete.zip
Taille : 8.21 Mo
